# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [31]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：data\E Commerce Dataset.xlsx
项目输出目录：c:\Users\Lenovo\Desktop\day4\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [32]:
# 在此写下你的答案：
# 1.每条记录代表一个电商平台用户的个人信息和行为数据，包括用户ID、登录设备、支付方式、订单偏好、使用时长、消费行为等特征。
# 2.项目的目标变量是 "Churn" 列，表示用户是否流失（1=流失，0=未流失）。
# 3.CustomerID 是用户的唯一标识符，是名义型变量而非连续数值。对其求平均值或进行数学运算没有业务意义，且会引入误导性分析结果。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [33]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().sum() / len(data) * 100).round(2),
        "唯一值数量": data.nunique()
    })
    return report

# TODO：生成清洗前质量报告
#quality_before = build_quality_report(raw_df)
#display(quality_before)

### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [34]:
# TODO：完成项目初始审计
print("=" * 50)
print("初始数据审计报告")
print("=" * 50)

print(f"原始数据形状：{raw_df.shape}")
print(f"完全重复行数：{raw_df.duplicated().sum()}")

customer_id_duplicates = raw_df["CustomerID"].duplicated().sum()
print(f"CustomerID 重复数量：{customer_id_duplicates}")
print("\nChurn 流失情况：")
print(raw_df["Churn"].value_counts())
churn_rate = (raw_df["Churn"].sum() / len(raw_df) * 100).round(2)
print(f"流失率：{churn_rate}%")

print("\n主要类别字段频数分布：")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())


# print("完全重复行数：", ...)
# print("CustomerID 重复数量：", ...)
# print(raw_df["Churn"].value_counts())
# print("流失率：", ...)
#
# for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
#     print(f"\n{col}")
#     print(raw_df[col].value_counts())

初始数据审计报告
原始数据形状：(5630, 20)
完全重复行数：0
CustomerID 重复数量：0

Churn 流失情况：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率：16.84%

主要类别字段频数分布：

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [35]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [36]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    cleaned_df = data.copy()
    logs = []
    # TODO：创建日志列表 logs
    logs = []
    # TODO：删除完全重复行，并记录日志
    initial_rows = len(cleaned_df)
    duplicate_count = cleaned_df.duplicated().sum()
    if duplicate_count > 0:
        cleaned_df = cleaned_df.drop_duplicates()
    logs.append({
        "步骤": "删除完全重复行",
        "处理规则": "删除所有字段完全相同的重复记录",
        "处理前记录数": initial_rows,
        "处理后记录数": len(cleaned_df),
        "影响记录数": initial_rows - len(cleaned_df)
    })
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        if col in cleaned_df.columns:
            missing_before = cleaned_df[col].isna().sum()
            if missing_before > 0:
                median_val = cleaned_df[col].median()
                cleaned_df[col] = cleaned_df[col].fillna(median_val)
                missing_after = cleaned_df[col].isna().sum()
                logs.append({
                    "步骤": f"填补缺失值 - {col}",
                    "处理规则": f"使用总体中位数 ({median_val:.2f}) 填补",
                    "处理前记录数": len(cleaned_df),
                    "处理后记录数": len(cleaned_df),
                    "影响记录数": missing_before - missing_after
                })
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col in cleaned_df.columns:
            for old_val, new_val in mapping.items():
                affected = (cleaned_df[col] == old_val).sum()
                if affected > 0:
                    cleaned_df[col] = cleaned_df[col].replace(old_val, new_val)
                    logs.append({
                        "步骤": f"类别标准化 - {col}",
                        "处理规则": f"将 '{old_val}' 统一为 '{new_val}'",
                        "处理前记录数": len(cleaned_df),
                        "处理后记录数": len(cleaned_df),
                        "影响记录数": affected
                    })
    # TODO：将 Churn 和 Complain 转为整数类型
    if "Churn" in cleaned_df.columns:
        cleaned_df["Churn"] = cleaned_df["Churn"].astype(int)
        logs.append({
            "步骤": "数据类型转换",
            "处理规则": "将 Churn 列转为整数类型",
            "处理前记录数": len(cleaned_df),
            "处理后记录数": len(cleaned_df),
            "影响记录数": len(cleaned_df)
        })
    if "Complain" in cleaned_df.columns:
        cleaned_df["Complain"] = cleaned_df["Complain"].astype(int)
        logs.append({
            "步骤": "数据类型转换",
            "处理规则": "将 Complain 列转为整数类型",
            "处理前记录数": len(cleaned_df),
            "处理后记录数": len(cleaned_df),
            "影响记录数": len(cleaned_df)
        })
    # TODO：返回 cleaned_df 与 cleaning_log
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log
    pass

### 任务 3：运行清洗函数并查看日志

In [37]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

print("=" * 50)
print("清洗日志")
print("=" * 50)
display(cleaning_log)

print("\n清洗后数据预览：")
display(cleaned_df.head())

print(f"\n清洗前形状：{raw_df.shape}")
print(f"清洗后形状：{cleaned_df.shape}")
# cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
#
# display(cleaning_log)
# cleaned_df.head()

清洗日志


,步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,删除所有字段完全相同的重复记录,5630,5630,0
1,填补缺失值 - Tenure,使用总体中位数 (9.00) 填补,5630,5630,264
2,填补缺失值 - WarehouseToHome,使用总体中位数 (14.00) 填补,5630,5630,251
3,填补缺失值 - HourSpendOnApp,使用总体中位数 (3.00) 填补,5630,5630,255
4,填补缺失值 - OrderAmountHikeFromlastYear,使用总体中位数 (15.00) 填补,5630,5630,265
5,填补缺失值 - CouponUsed,使用总体中位数 (1.00) 填补,5630,5630,256
6,填补缺失值 - OrderCount,使用总体中位数 (2.00) 填补,5630,5630,258
7,填补缺失值 - DaySinceLastOrder,使用总体中位数 (3.00) 填补,5630,5630,307
8,类别标准化 - PreferredLoginDevice,将 'Phone' 统一为 'Mobile Phone',5630,5630,1231
9,类别标准化 - PreferredPaymentMode,将 'COD' 统一为 'Cash on Delivery',5630,5630,365



清洗后数据预览：


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60



清洗前形状：(5630, 20)
清洗后形状：(5630, 20)


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [38]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [0, 12, 24, 36, 48, 60, float('inf')]
tenure_labels = ["0-1年", "1-2年", "2-3年", "3-4年", "4-5年", "5年以上"]
cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    right=False
)
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)
# TODO：生成 outlier_report（每行对应一个待检查字段）
outlier_fields = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_data = []

for field in outlier_fields:
    if field in cleaned_df.columns:
        summary = iqr_outlier_summary(cleaned_df[field])
        summary["字段名"] = field
        outlier_data.append(summary)

outlier_report = pd.DataFrame(outlier_data)
outlier_report = outlier_report[["字段名", "Q1", "Q3", "下限", "上限", "候选异常值数量"]]

print("候选异常值报告：")
display(outlier_report)

候选异常值报告：


,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [39]:
# TODO：完成业务规则检查
rules = [
    ("Tenure >= 0", "Tenure", lambda x: x < 0),
    ("WarehouseToHome >= 0", "WarehouseToHome", lambda x: x < 0),
    ("OrderCount > 0", "OrderCount", lambda x: x <= 0),
    ("CashbackAmount >= 0", "CashbackAmount", lambda x: x < 0)
]

business_rule_results = []
for rule_name, col, condition in rules:
    if col in cleaned_df.columns:
        invalid_count = cleaned_df[condition(cleaned_df[col])].shape[0]
        business_rule_results.append({
            "规则": rule_name,
            "检查字段": col,
            "不合规记录数": invalid_count
        })
business_rule_report = pd.DataFrame(business_rule_results)

print("业务规则检查报告：")
display(business_rule_report)

# 处理结论
print("\n处理结论：")
if business_rule_report["不合规记录数"].sum() == 0:
    print("所有业务规则检查通过，未发现不合规记录。数据质量良好。")
else:
    print("发现不合规记录，建议进一步核查数据来源和业务含义。")
    print("本项目不自动删除异常值，仅记录供后续分析参考。")
# business_rule_report = pd.DataFrame({
#     "规则": [...],
#     "不合规记录数": [...]
# })
# display(business_rule_report)
#
# 处理结论：

业务规则检查报告：


,规则,检查字段,不合规记录数
0,Tenure >= 0,Tenure,0
1,WarehouseToHome >= 0,WarehouseToHome,0
2,OrderCount > 0,OrderCount,0
3,CashbackAmount >= 0,CashbackAmount,0



处理结论：
所有业务规则检查通过，未发现不合规记录。数据质量良好。


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [40]:
# TODO：完成最终验收
quality_before = build_quality_report(raw_df)
quality_after = build_quality_report(cleaned_df)

# 验收断言
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "Phone 未标准化"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "COD 未标准化"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "CC 未标准化"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "缺少新增特征列"

print("=" * 50)
print("验收通过！所有断言已满足。")
print("=" * 50)
# quality_after = build_quality_report(cleaned_df)
#
# assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
# assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
# assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
# assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
# assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
#
# TODO：导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
# quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
# quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
# cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
# cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
#
# TODO：输出 outlier_report 和 business_rule_report
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")
# TODO：输出交付文件的路径
print("\n交付文件清单：")
print("-" * 50)
for file in OUTPUT_DIR.iterdir():
    if file.is_file():
        size_kb = file.stat().st_size / 1024
        print(f"✓ {file.name} ({size_kb:.1f} KB)")

print(f"\n所有文件已保存至：{OUTPUT_DIR}")

验收通过！所有断言已满足。

交付文件清单：
--------------------------------------------------
✓ business_rule_report.csv (0.2 KB)
✓ cleaning_log.csv (1.2 KB)
✓ data_quality_after.csv (0.7 KB)
✓ data_quality_before.csv (0.7 KB)
✓ ecommerce_customer_cleaned.csv (641.4 KB)
✓ outlier_report.csv (0.2 KB)

所有文件已保存至：c:\Users\Lenovo\Desktop\day4\output\day04_project


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？